## 1. Preparando el Entorno Cuántico

Antes de empezar, necesitamos cargar nuestras herramientas.
En este caso vamos a usar **Qiskit** (el SDK cuántico de IBM) para simular un procesador cuántico real en nuestra máquina. También usaremos `numpy` para generar los arrays de números aleatorios que formarán nuestra clave.

* **El Simulador:** Usaremos `AerSimulator`, que actuará como nuestro hardware cuántico virtual.
* **N_BITS:** Definiremos la longitud inicial del mensaje que Alice va a intentar enviar.

In [ ]:
!pip install qiskit qiskit-aer

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import numpy as np

# Simulador cuántico local
simulador = AerSimulator()

# Longitud del mensaje original que Alice mandara
N_BITS = 20

print("Entorno cuántico preparado.")

Entorno cuántico preparado.


## 2. Alice prepara la transmisión

Alice quiere enviarle una clave secreta a Bob, pero sabe que el canal (la fibra óptica) puede estar pinchado. Por eso, usa el protocolo cuántico BB84:

1. **Genera datos clásicos:** Crea un array de ceros y unos totalmente aleatorios (su mensaje).
2. **Genera bases aleatorias:** Para cada bit, decide al azar si lo va a codificar en la **Base Estándar (0)** o en la **Base Diagonal (1)**.
3. **Codificación Cuántica:**
   * Si el bit es `1`, aplica una **puerta X** (equivalente a un NOT clásico).
   * Si la base es `1` (Diagonal), aplica una **puerta Hadamard (H)**. Esto pone al qubit en *superposición*, ocultando su valor real a menos que se lea con la base correcta.:

In [ ]:
# 1. Generación de datos clásicos (Arrays de 0s y 1s)
alice_bits = np.random.randint(2, size=N_BITS)
alice_bases = np.random.randint(2, size=N_BITS)

print(f"Bits secretos de Alice: {alice_bits}")
print(f"Bases usadas por Alice: {alice_bases} (0=Z, 1=X)")

# 2. Alice codifica la información en Qubits
circuitos_cuanticos = []

for i in range(N_BITS):
    # Creamos un circuito con 1 qubit y 1 bit clásico para la lectura final
    qc = QuantumCircuit(1, 1)

    # Si el bit es 1, aplicamos la puerta X (equivalente a un NOT clásico)
    if alice_bits[i] == 1:
        qc.x(0)

    # Si la base es 1 (Diagonal), aplicamos la puerta Hadamard (H) para crear superposición
    if alice_bases[i] == 1:
        qc.h(0)

    circuitos_cuanticos.append(qc)

print("Qubits codificados y enviados")

Bits secretos de Alice: [0 0 1 1 1 1 1 0 0 0 1 0 0 0 0 1 0 1 0 0]
Bases usadas por Alice: [0 0 1 0 0 1 0 0 0 0 0 0 1 1 1 1 0 1 0 1] (0=Z, 1=X)
Qubits codificados y enviados


## 3. ACTIVIDAD: Intercepción del mensaje

Imagina que sois la hacker Eve. Habéis pinchado el cable y los qubits de Alice están pasando por vuestras manos. Queréis leerlos y enviárselos a Bob sin que se entere.

**Vuestro problema:** No sabéis qué base usó Alice para cada qubit.
**Vuestra solución:** Generáis vuestro propio array de bases aleatorias (`eve_bases`).

**Instrucciones para el código:**
Debéis completar el bucle `for` para interceptar cada qubit:
1. Si vuestra base aleatoria es `1` (Diagonal), aplicad una puerta Hadamard (`qc.h(0)`) para intentar "traducir" el qubit antes de leerlo.
2. Medid el qubit usando `qc.measure(0, 0)` para extraer el dato clásico.

*Nota física:* Al medir el qubit, colapsáis su estado. Si adivinasteis mal la base, habréis destruido el qubit original de Alice y Bob recibirá "basura cuántica".

In [ ]:
# Eve intercepta el canal y elige sus propias bases al azar
eve_bases = np.random.randint(2, size=N_BITS)
eve_bits_robados = []

# --- INICIO DE LA ACTIVIDAD INTERACTIVA ---
# Recorrer el vector (N_BITS) en un bucle.

    # Coger el circuito del vector


    # Coger el circuito y aplicar una puerta
    # En este caso si el qubit es un 1, aplicamos la base diagonal

         # Aplicar una puerta Hadamard (h) para utilizar la base diagonal.

    # Medir el qubits 0 y guardar el resultado en el bit clasico 0


    # Simulamos la medición de Eve
    resultado_eve = simulador.run(qc, shots=1).result()
    bit_leido = int(list(resultado_eve.get_counts().keys())[0])
    eve_bits_robados.append(bit_leido)

# --- FIN DE LA ACTIVIDAD INTERACTIVA ---

print(f"Bases elegidas por la espía: {eve_bases}")
print(f"Bits que Eve cree haber robado: {eve_bits_robados}")

Bases elegidas por la espía: [1 1 0 0 0 1 1 1 0 1 0 1 0 1 1 1 1 0 1 0]
Bits que Eve cree haber robado: [0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0]


## 4. Bob recibe el mensaje a ciegas

Bob recibe los qubits. Él no sabe si Eve los ha manipulado o no.
Al igual que hizo Eve, Bob **tampoco sabe** qué bases usó Alice originalmente. Así que hace exactamente lo mismo: genera su propio array de bases aleatorias (`bob_bases`).

* Si Bob elige la base Diagonal, aplica la puerta `H`.
* Finalmente, mide el qubit para obtener un bit clásico (un `0` o un `1`).

In [ ]:
bob_bases = np.random.randint(2, size=N_BITS)
bob_bits = []

for i in range(N_BITS):
    qc = circuitos_cuanticos[i] # Bob coge el qubit que le llega

    # Bob ajusta su lector según su base elegida al azar
    if bob_bases[i] == 1:
        qc.h(0) # Vuelve a aplicar Hadamard si elige base diagonal

    # Bob mide (si Eve ya midió, este qubit ya no es el original)
    qc.measure(0, 0)

    resultado_bob = simulador.run(qc, shots=1).result()
    bit_leido = int(list(resultado_bob.get_counts().keys())[0])
    bob_bits.append(bit_leido)

print(f"Bases de Bob: {bob_bases}")
print(f"Bits leídos por Bob: {bob_bits}")

Bases de Bob: [0 1 1 0 1 0 1 1 1 1 1 1 0 0 0 1 0 1 1 1]
Bits leídos por Bob: [1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0]


## 5. El "Sifting"

Ya no hay más qubits. Ahora Alice y Bob se conectan por un chat clásico público (como WhatsApp o un canal de Discord). Todo el mundo puede leer lo que dicen, incluida Eve.

**Fase 1: El Filtro (Sifting)**
Solo se dicen las **bases** que usaron (los arrays de 0s y 1s), ¡nunca los valores medidos! Si para el índice `i` usaron la misma base, se guardan ese bit. Si usaron bases distintas, tiran ese bit a la basura porque la lectura no es fiable.

**Fase 2: El Sacrificio (Detección de intrusos)**
Para saber si Eve ha estado escuchando, cogen una pequeña muestra de su clave ya filtrada (por ejemplo, los primeros 4 bits) y **se los dicen en voz alta**.
* Si coinciden al 100%, saben matemáticamente que nadie ha mirado los qubits. Esos 4 bits se sacrifican (ya no son secretos), pero el resto de la clave es 100% segura.
* Si hay un error de ~25%, significa que Eve midió los qubits por el camino y alteró sus estados físicos. ¡La clave se descarta!

In [ ]:
# 1. El filtro clásico (Sifting)
clave_alice = []
clave_bob = []

for i in range(N_BITS):
    if alice_bases[i] == bob_bases[i]:
        clave_alice.append(alice_bits[i])
        clave_bob.append(bob_bits[i])

print(f"Longitud de la clave tras descartar bases distintas: {len(clave_alice)} bits")

# 2. Comprobación de seguridad
errores = 0
for i in range(len(clave_alice)):
    if clave_alice[i] != clave_bob[i]:
        errores += 1

tasa_error = errores / len(clave_alice)

print(f"\n--- INFORME DE SEGURIDAD ---")
print(f"Clave esperada por Alice: {clave_alice}")
print(f"Clave obtenida por Bob:   {clave_bob}")
print(f"Tasa de error detectada:  {tasa_error * 100}%\n")

if tasa_error > 0.05: # Tolerancia del 5%
    print("Se ha detectado una perturbación en la contraseña. Se debe de descartar la contraseña.")
else:
    print("Clave validada y lista para usarse.")

Longitud de la clave tras descartar bases distintas: 7 bits

--- INFORME DE SEGURIDAD ---
Clave esperada por Alice: [np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0)]
Clave obtenida por Bob:   [1, 0, 1, 1, 0, 1, 0]
Tasa de error detectada:  28.57142857142857%

Se ha detectado una perturbación en la contraseña. Se debe de descartar la contraseña.
